In [4]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sentence_transformers import SentenceTransformer
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import dagshub
import numpy as np

In [5]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/sentiment_analysis/cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [ ]:
dagshub.init(repo_owner= 'vaibhav-patel01' , repo_name= 'YT_Comments_Analysis_chrome_extension' , mlflow= True)

mlflow.set_tracking_uri("https://dagshub.com/vaibhav-patel01/YT_Comments_Analysis_chrome_extension.mlflow")
mlflow.set_experiment("finding_best_feature_engineering_algo")

Initialized MLflow to track repo "vaibhav-patel01/YT_Comments_Analysis_chrome_extension"

Repository vaibhav-patel01/YT_Comments_Analysis_chrome_extension initialized!

<Experiment: artifact_location='mlflow-artifacts:/cf5821c3c16a404c81ff005e5f1f60f0', creation_time=1783399519152, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783399519152, lifecycle_stage='active', name='finding_best_feature_engineering_algo', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
comments = df["CommentText"].tolist()
model = SentenceTransformer("all-MiniLM-L6-v2")

# GPU Encoding: Fast, simple, and has a native progress bar
comments_embeddings = model.encode(
    comments,
    batch_size=256,          # Bump this up from 64 to 256 to feed the GPU faster
    show_progress_bar=True,  # This works perfectly here
    convert_to_numpy=True
).astype(np.float32)

# Save to disk

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3710 [00:00<?, ?it/s]

In [ ]:
np.save("/content/drive/MyDrive/sentiment_analysis/comments_embeddings.npy", comments_embeddings)

In [ ]:
loaded_embeddings = np.load("/content/drive/MyDrive/sentiment_analysis/comments_embeddings.npy")

In [ ]:
print(df.shape)
print(loaded_embeddings.shape)

(949621, 2)
(949621, 384)


In [ ]:
# Step 2: Remove rows where the target labels (category) are NaN
warnings.filterwarnings("ignore", category=UserWarning)
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})
X = loaded_embeddings
y = df["Sentiment"]

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_embeddings")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: embeddings, model={model_name}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# Step 7: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LightGBM"
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-09 06:23:55,459] A new study created in memory with name: no-name-68a464d3-b4b4-447a-9825-9d7c327619bd


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 5.091768 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 759696, number of used features: 384
[LightGBM] [Info] Start training from score -1.098565
[LightGBM] [Info] Start training from score -1.098652
[LightGBM] [Info] Start training from score -1.098620


[I 2026-07-09 06:29:52,563] Trial 0 finished with value: 0.5228958799526129 and parameters: {'n_estimators': 50, 'learning_rate': 0.001283780277824115, 'max_depth': 7}. Best is trial 0 with value: 0.5228958799526129.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 4.794993 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 759696, number of used features: 384
[LightGBM] [Info] Start training from score -1.098565
[LightGBM] [Info] Start training from score -1.098652
[LightGBM] [Info] Start training from score -1.098620


[I 2026-07-09 06:35:29,553] Trial 1 finished with value: 0.5204475450835856 and parameters: {'n_estimators': 59, 'learning_rate': 0.0002022432764941344, 'max_depth': 5}. Best is trial 0 with value: 0.5228958799526129.


...

No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7f6437bc22a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7f6437bc22a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
